In [8]:
import pandas as pd
import numpy as np
import glob
import os

df = pd.read_csv("202301_Eng.csv", skiprows=7)
print(df.head())
print(df.columns)

         Date Hour Central/Western Southern Eastern Kwun Tong Sham Shui Po  \
0  2023-01-01   01               3        3       4         3            3   
1         NaN   02               3        3       4         3            3   
2         NaN   03               3        3       3         3            3   
3         NaN   04               3        3       3         3            3   
4         NaN   05               3        3       3         3            3   

  Kwai Chung Tsuen Wan Tseung Kwan O Yuen Long Tuen Mun Tung Chung Tai Po  \
0          3         3             3         3        4          3      3   
1          3         3             3         3        3          2      3   
2          3         3             3         3        3          2      3   
3          3         3             3         3        3          2      3   
4          3         3             3         3        3          2      3   

  Sha Tin North Tap Mun Causeway Bay Central Mong Kok  
0       3   

In [9]:
def process_aqhi_month(file_path, encoding="utf-8"):
    # read csv
    df = pd.read_csv(file_path, skiprows=7, encoding=encoding)
    
    # clean column names
    df.columns = [str(c).strip() for c in df.columns]
    
    # keep valid hour rows only
    df["Hour"] = pd.to_numeric(df["Hour"], errors="coerce")
    df = df[df["Hour"].notna()].copy()
    
    # forward fill date
    df["Date"] = df["Date"].ffill()
    
    # reshape wide -> long
    long_df = df.melt(
        id_vars=["Date", "Hour"],
        var_name="station",
        value_name="aqhi_raw"
    )
    
    # detect asterisk-marked values before cleaning
    long_df["has_asterisk"] = (
        long_df["aqhi_raw"]
        .astype(str)
        .str.contains(r"\*", regex=True, na=False)
    )
    
    # clean AQHI
    long_df["aqhi"] = (
        long_df["aqhi_raw"]
        .astype(str)
        .str.strip()
        .str.replace("*", "", regex=False)
    )
    long_df["aqhi"] = long_df["aqhi"].replace(["", "nan", "None", "-"], pd.NA)
    long_df["aqhi"] = pd.to_numeric(long_df["aqhi"], errors="coerce")
    
    # parse date/hour
    long_df["date"] = pd.to_datetime(long_df["Date"], errors="coerce")
    long_df["hour_index"] = pd.to_numeric(long_df["Hour"], errors="coerce")
    
    # clean station names
    long_df["station"] = long_df["station"].astype(str).str.strip()
    
    # source file
    long_df["source_file"] = os.path.basename(file_path)
    
    # final columns
    long_df = long_df[
        ["date", "hour_index", "station", "aqhi", "has_asterisk", "source_file"]
    ].copy()
    
    return long_df

In [10]:
jan = process_aqhi_month("202301_Eng.csv", encoding="utf-8")
jan.head(20)


,date,hour_index,station,aqhi,has_asterisk,source_file
0,2023-01-01,1.0,Central/Western,3.0,False,202301_Eng.csv
1,2023-01-01,2.0,Central/Western,3.0,False,202301_Eng.csv
2,2023-01-01,3.0,Central/Western,3.0,False,202301_Eng.csv
3,2023-01-01,4.0,Central/Western,3.0,False,202301_Eng.csv
4,2023-01-01,5.0,Central/Western,3.0,False,202301_Eng.csv
5,2023-01-01,6.0,Central/Western,3.0,False,202301_Eng.csv
6,2023-01-01,7.0,Central/Western,3.0,False,202301_Eng.csv
7,2023-01-01,8.0,Central/Western,3.0,False,202301_Eng.csv
8,2023-01-01,9.0,Central/Western,3.0,False,202301_Eng.csv
9,2023-01-01,10.0,Central/Western,3.0,False,202301_Eng.csv


In [11]:
jan_daily = (
    jan.groupby(["date", "station"], as_index=False)
    .agg(
        daily_max_aqhi=("aqhi", "max"),
        daily_mean_aqhi=("aqhi", "mean"),
        hourly_count=("aqhi", "count"),
        starred_hours=("has_asterisk", "sum")
    )
)

jan_daily.head(20)

,date,station,daily_max_aqhi,daily_mean_aqhi,hourly_count,starred_hours
0,2023-01-01,Causeway Bay,4.0,3.541667,24,0
1,2023-01-01,Central,4.0,3.250000,24,0
2,2023-01-01,Central/Western,4.0,3.166667,24,0
3,2023-01-01,Eastern,4.0,3.458333,24,0
4,2023-01-01,Kwai Chung,3.0,3.000000,24,0
5,2023-01-01,Kwun Tong,4.0,3.083333,24,0
6,2023-01-01,Mong Kok,4.0,3.458333,24,0
7,2023-01-01,North,3.0,2.875000,24,0
8,2023-01-01,Sha Tin,4.0,3.041667,24,0
9,2023-01-01,Sham Shui Po,4.0,3.125000,24,0
